# 09. Statistical Evaluation: TRTR vs TSTR Benchmark

This notebook provides rigorous statistical comparison between:
- **Model A (TRTR)**: Train on Real data, Test on Real data (Baseline)
- **Model B (TSTR)**: Train on Synthetic data, Test on Real data (CVAE-augmented)

## Statistical Design

- **Patient-Level Stratification**: Bootstrap samples patients, not spectra
- **Honest Predictions**: Each patient predicted ONLY by models from its validation fold
- **No Data Leakage**: Fold i models never predict on fold i's patients (they trained on them)
- **Cached Predictions**: Predictions computed once, reused across bootstrap iterations
- **Resampling**: Stratified bootstrap at patient level (cluster bootstrap)
- **Test**: Wilcoxon signed-rank test on bootstrap differences
- **Multiple testing**: Bonferroni correction across 4 models
- **Metrics**: Accuracy, F1-macro, Cohen's κ, McNemar test

## Section 1: Imports & Setup

In [ ]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import tensorflow as tf
from datetime import datetime
from pathlib import Path
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    cohen_kappa_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment
from src.data.loader import GliomaDataLoader

print("✓ Imports successful")

In [2]:
# Load configuration
config_name = 'exp_105'
config, output_dir = setup_experiment(config_name)

CFG = {
    "data_path": config['paths']['processed_file'],
    "split_file": config['paths']['splits_file'],
    "model_a_dir": config['paths']['models_baseline_dir'],  # TRTR
    "model_b_dir": config['paths']['models_tetr_dir'],      # TSTR
    "output_dir": config['paths']['statistical_dir'],
    "n_bootstrap": config['statistics']['bootstrap_samples'],
    "alpha": config['statistics']['alpha'],
    "random_seed": config['statistics']['random_seed'],
    "models": {
        "Apollo": {"prefix": "apollo", "ext": ".joblib", "type": "sklearn"},
        "RF_Standard": {"prefix": "rf", "ext": ".joblib", "type": "sklearn"},
        "XGBoost": {"prefix": "xgb", "ext": ".json", "type": "xgboost"},
        "1D_CNN": {"prefix": "cnn", "ext": ".keras", "type": "keras"},
    },
    "primary_metric": config['statistics']['primary_metric'],
    "n_folds": config['cv_strategy']['n_folds']
}
config["cv_strategy"]["patient_cap"] = None
np.random.seed(CFG["random_seed"])
os.makedirs(CFG["output_dir"], exist_ok=True)

print(f"✓ Configuration loaded: {config_name}")
print(f"  Bootstrap samples: {CFG['n_bootstrap']}")
print(f"  α = {CFG['alpha']}")
print(f"  Number of folds: {CFG['n_folds']}")


EXPERIMENT SETUP

Working directory: c:\Users\Iulian\Documents\glioma\glioma-synthetic-data

 Loaded configuration: exp_105
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Iulian\Documents\glioma\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Configuration loaded: exp_105
  Bootstrap samples: 1000
  α = 0.05
  Number of folds: 5


## Section 2: Load Data & Build Patient-to-Fold Mapping

In [3]:
# Load data
print("Loading data...")
loader = GliomaDataLoader.from_config(
    h5_data_path=CFG['data_path'],
    config=config,
    memory_mapped=False
)

# ---------------------------------------------------------
# SECTION 2: Extract Pure Iterator Data (Perfectly Scaled)
# ---------------------------------------------------------
with open(CFG['split_file'], 'r') as f:
    splits_data = json.load(f)
splits = splits_data.get('cv_folds', splits_data)

print("\nExtracting preprocessed validation data directly from iterator...")
loader_iter = iter(loader)
all_groups = np.array(loader.groups)

X_val_list = []
y_val_list = []
patient_ids_list = []

num_classes = config['models']['cnn']['num_classes']


for fold_idx in range(CFG['n_folds']):
    fold_data = next(loader_iter)
    fold_name = fold_data['fold_name']
    
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1].copy()
    
    # Replicate Notebook 02 label mapping to ensure matching classes
    if np.min(y_val) == 1:
        y_val = y_val - 1
    
    # Apply binary conversion if needed (for exp_105)
    if num_classes == 2:
        y_val = np.where(y_val < 3, 0, 1)

    # Get patient IDs using the string lookup
    raw_string_ids = splits[fold_name]['test']
    val_indices_sorted = np.where(np.isin(all_groups, raw_string_ids))[0]
    fold_sample_ids = all_groups[val_indices_sorted]
    fold_patient_ids = np.array([s.split('_')[0] + '_' + s.split('_')[1] for s in fold_sample_ids])
    
    X_val_list.append(X_val)
    y_val_list.append(y_val)
    patient_ids_list.append(fold_patient_ids)
    print(f"  ✓ {fold_name}: {len(y_val)} spectra mapped to {len(np.unique(fold_patient_ids))} patients")

# Combine into unified master arrays for the bootstrap
y_all = np.concatenate(y_val_list)
patient_ids_all = np.concatenate(patient_ids_list)
print(f"\n✓ Unified Test Set: {len(y_all)} spectra ready for evaluation.")
if num_classes == 2:
    print(f"  ✓ Binary classification task detected (Classes: {np.unique(y_all)})")

Loading data...
Loading CV splits from: data/splits/strict_splits.json
Loading data from results\exp_102\data\glioma_dataset.h5...
[*] 'tumor_only=True': Filtering out healthy tissue...
    Filtered to 226284 Pure Tumor Spectra.
[OK] Spectral Crop: [700, 1800] cm-1 -> indices [337, 908]
  Wavenumber mapping: 699.91 -> 1799.14 cm-1
  Renormalizing to cropped region...
    After renormalization: min=0.000000, max=1.000000 (fingerprint region)
  Spectra shape after crop: (226284, 571)

Extracting preprocessed validation data directly from iterator...

 Using CV splits from: c:\Users\Iulian\Documents\glioma\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples
  ✓ fold_0: 45936 spectra mapped to 10 patients
   Fold 1: 163393 train samples, 42627 val samples
  ✓ fold_1: 42627 spectra mappe

## Section 3: Helper Functions

In [4]:
def find_model_path(base_dir: str, prefix: str, ext: str, fold_idx: int) -> str:
    """Find model file for a specific fold."""
    candidates = [
        f"{base_dir}/{prefix}_fold_{fold_idx}{ext}",
        f"{base_dir}/{prefix}_fold{fold_idx}{ext}",
        f"{base_dir}/{prefix}_{fold_idx}{ext}",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Model '{prefix}' fold {fold_idx} not found")


def load_and_predict(path: str, model_type: str, X: np.ndarray) -> np.ndarray:
    """Load model and return predictions (class labels)."""
    if model_type == "sklearn":
        model = joblib.load(path)
        return model.predict(X).astype(int)
    
    elif model_type == "xgboost":
        model = xgb.XGBClassifier()
        model.load_model(path)
        return model.predict(X).astype(int)
    
    elif model_type == "keras":
        X_3d = X.reshape(-1, X.shape[1], 1)
        try:
            model = tf.keras.models.load_model(path, compile=False)
            preds = np.argmax(model.predict(X_3d, verbose=0), axis=1)
            tf.keras.backend.clear_session()
            return preds.astype(int)
        except (TypeError, ValueError) as e:
            # Handle deprecated BatchNormalization parameters in older saved models
            if "renorm" in str(e) or "BatchNormalization" in str(e):
                raise FileNotFoundError(f"Keras model has incompatible BatchNormalization config: {str(e)[:100]}")
            else:
                raise
    else:
        raise ValueError(f"Unknown model type: '{model_type}'")

In [6]:
def stratified_patient_bootstrap(preds_A, preds_B, y_true, patient_ids, n_boot=1000, seed=42):
    """
    Perform patient-level stratified bootstrap for comparing two models.
    
    Resamples patients (not spectra) to maintain patient clustering.
    Returns bootstrap distributions of accuracy for both models.
    
    Parameters:
    -----------
    preds_A : np.ndarray
        Predictions from Model A (shape: n_samples)
    preds_B : np.ndarray
        Predictions from Model B (shape: n_samples)
    y_true : np.ndarray
        True labels (shape: n_samples)
    patient_ids : np.ndarray
        Patient ID for each sample (shape: n_samples)
    n_boot : int
        Number of bootstrap iterations (default: 1000)
    seed : int
        Random seed for reproducibility (default: 42)
    
    Returns:
    --------
    boot_A : np.ndarray
        Bootstrap accuracy distribution for Model A (shape: n_boot)
    boot_B : np.ndarray
        Bootstrap accuracy distribution for Model B (shape: n_boot)
    boot_diffs : np.ndarray
        Bootstrap differences (B - A) (shape: n_boot)
    """
    rng = np.random.RandomState(seed)
    
    boot_A = np.zeros(n_boot)
    boot_B = np.zeros(n_boot)
    
    # Get unique patient IDs and create mapping
    unique_patients = np.unique(patient_ids)
    n_patients = len(unique_patients)
    
    for i in range(n_boot):
        # Sample patients with replacement
        sampled_patients = rng.choice(unique_patients, size=n_patients, replace=True)
        
        # Get indices for sampled patients
        boot_indices = np.concatenate([
            np.where(patient_ids == pid)[0] for pid in sampled_patients
        ])
        
        # Compute accuracy on bootstrap sample
        boot_A[i] = accuracy_score(y_true[boot_indices], preds_A[boot_indices])
        boot_B[i] = accuracy_score(y_true[boot_indices], preds_B[boot_indices])
    
    boot_diffs = boot_B - boot_A
    
    return boot_A, boot_B, boot_diffs

In [ ]:
# ── New helper functions: proba loading, full metrics, slide aggregation ────

def load_and_predict_proba(path: str, model_type: str, X: np.ndarray) -> np.ndarray:
    """Load model and return predicted probability for the positive class (index 1)."""
    if model_type == "sklearn":
        model = joblib.load(path)
        return model.predict_proba(X)[:, 1].astype(np.float32)
    elif model_type == "xgboost":
        model = xgb.XGBClassifier()
        model.load_model(path)
        return model.predict_proba(X)[:, 1].astype(np.float32)
    elif model_type == "keras":
        X_3d = X.reshape(-1, X.shape[1], 1)
        try:
            model = tf.keras.models.load_model(path, compile=False)
            preds = model.predict(X_3d, verbose=0)
            tf.keras.backend.clear_session()
            return preds[:, 1].astype(np.float32)
        except (TypeError, ValueError) as e:
            if "renorm" in str(e) or "BatchNormalization" in str(e):
                raise FileNotFoundError(f"Keras model incompatible: {str(e)[:100]}")
            raise
    else:
        raise ValueError(f"Unknown model type: '{model_type}'")


def compute_full_metrics(y_true: np.ndarray, preds: np.ndarray,
                          proba: np.ndarray = None) -> dict:
    """
    Full evaluation metric set for binary IDH classification.
    Returns a flat dict suitable for JSON serialisation.
    Includes: accuracy, balanced accuracy, macro/weighted F1, precision, recall,
    Cohen's kappa, per-class breakdown, confusion matrix,
    sensitivity, specificity, AUROC, AUPRC (last two require proba).
    """
    m = {}
    m['accuracy']          = float(accuracy_score(y_true, preds))
    m['balanced_accuracy'] = float(balanced_accuracy_score(y_true, preds))
    m['f1_macro']          = float(f1_score(y_true, preds, average='macro',    zero_division=0))
    m['f1_weighted']       = float(f1_score(y_true, preds, average='weighted', zero_division=0))
    m['precision_macro']   = float(precision_score(y_true, preds, average='macro',    zero_division=0))
    m['recall_macro']      = float(recall_score(y_true, preds, average='macro',       zero_division=0))
    m['cohen_kappa']       = float(cohen_kappa_score(y_true, preds))

    # Per-class breakdown (class 0 = Wildtype, class 1 = Mutant)
    classes = sorted(np.unique(y_true).tolist())
    f1_pc   = f1_score(y_true, preds, average=None, labels=classes, zero_division=0)
    prec_pc = precision_score(y_true, preds, average=None, labels=classes, zero_division=0)
    rec_pc  = recall_score(y_true, preds, average=None, labels=classes, zero_division=0)
    for i, c in enumerate(classes):
        m[f'f1_class{c}']        = float(f1_pc[i])
        m[f'precision_class{c}'] = float(prec_pc[i])
        m[f'recall_class{c}']    = float(rec_pc[i])

    # Confusion matrix + sensitivity / specificity (binary)
    cm = confusion_matrix(y_true, preds, labels=classes)
    m['confusion_matrix'] = cm.tolist()
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        m['sensitivity'] = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
        m['specificity'] = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0

    # AUROC + AUPRC (require probabilities for positive class)
    if proba is not None:
        m['auroc'] = float(roc_auc_score(y_true, proba))
        m['auprc'] = float(average_precision_score(y_true, proba))

    return m


def aggregate_by_patient(preds: np.ndarray, y_true: np.ndarray,
                          patient_ids: np.ndarray,
                          proba: np.ndarray = None) -> pd.DataFrame:
    """
    Aggregate spectrum-level predictions to patient/slide level.

    Methods:
        majority_voting  – most frequent predicted class per patient
        mean_probability – mean proba across spectra then argmax (requires proba)

    Returns a DataFrame with:
        slide_id, true_label, pred_majority[, mean_proba, pred_mean_prob]
    """
    df = pd.DataFrame({
        'slide_id':   patient_ids,
        'true_label': y_true,
        'pred':       preds,
    })
    if proba is not None:
        df['proba'] = proba

    rows = []
    for sid, grp in df.groupby('slide_id'):
        row = {
            'slide_id':      sid,
            'true_label':    int(grp['true_label'].mode()[0]),
            'pred_majority': int(grp['pred'].mode()[0]),
        }
        if proba is not None:
            mp = float(grp['proba'].mean())
            row['mean_proba']     = mp
            row['pred_mean_prob'] = int(mp >= 0.5)
        rows.append(row)

    return pd.DataFrame(rows)


def compute_slide_metrics(slide_df: pd.DataFrame) -> dict:
    """Accuracy and Macro-F1 for both patient-level aggregation methods."""
    y_true = slide_df['true_label'].values
    m = {}

    y_maj = slide_df['pred_majority'].values
    m['majority_accuracy'] = float(accuracy_score(y_true, y_maj))
    m['majority_f1_macro'] = float(f1_score(y_true, y_maj, average='macro', zero_division=0))

    if 'pred_mean_prob' in slide_df.columns:
        y_mp = slide_df['pred_mean_prob'].values
        m['mean_prob_accuracy'] = float(accuracy_score(y_true, y_mp))
        m['mean_prob_f1_macro'] = float(f1_score(y_true, y_mp, average='macro', zero_division=0))

    return m


print("✓ Helper functions defined")

## Section 5: Statistical Evaluation Loop

In [ ]:
# ---------------------------------------------------------
# SECTION 5: Fast Statistical Evaluation (No Data Leakage)
# ---------------------------------------------------------
results_all = []
all_preds_store = {}   # keyed by model_name; reused by slide-level aggregation below
alpha_corrected = CFG['alpha'] / len(CFG['models'])
n_folds = CFG['n_folds']

print(f"\n{'='*70}")
print(f"STATISTICAL EVALUATION: TRTR vs TSTR (Patient-Level Bootstrap)")
print(f"{'='*70}")
print(f"Bonferroni-corrected α = {alpha_corrected:.4f}\n")

for model_name, m_cfg in CFG['models'].items():
    print(f"\n{'-'*70}")
    print(f"MODEL: {model_name}")
    print(f"{'-'*70}")
    
    preds_A_list = []
    preds_B_list = []
    proba_A_list = []
    proba_B_list = []
    skip_model = False
    
    # 1. Generate strictly unseen predictions fold-by-fold
    for fold_idx in range(n_folds):
        if skip_model:
            break
            
        X_val = X_val_list[fold_idx]
        
        try:
            # TRTR (Model A)
            path_A = find_model_path(CFG['model_a_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_A_list.append(load_and_predict(path_A, m_cfg['type'], X_val))
            proba_A_list.append(load_and_predict_proba(path_A, m_cfg['type'], X_val))
            
            # TSTR (Model B)
            path_B = find_model_path(CFG['model_b_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_B_list.append(load_and_predict(path_B, m_cfg['type'], X_val))
            proba_B_list.append(load_and_predict_proba(path_B, m_cfg['type'], X_val))
        except FileNotFoundError as e:
            if "incompatible" in str(e).lower() or "batchnormalization" in str(e).lower():
                print(f"  ⚠️  {model_name}: Keras models incompatible (deprecated BatchNormalization). Skipping.")
                skip_model = True
            else:
                print(f"  ✗ Missing model for {model_name} fold {fold_idx}")

    if len(preds_A_list) < n_folds or len(preds_B_list) < n_folds:
        print(f"⚠️  {model_name}: Incomplete predictions ({len(preds_A_list)}/{n_folds} folds). Skipping.")
        continue

    # 2. Concatenate into full unseen prediction arrays
    preds_A = np.concatenate(preds_A_list)
    preds_B = np.concatenate(preds_B_list)
    proba_A = np.concatenate(proba_A_list)
    proba_B = np.concatenate(proba_B_list)
    has_proba = not np.any(np.isnan(proba_A))

    # Store arrays for slide-level aggregation (avoids reloading models)
    all_preds_store[model_name] = {
        'preds_A': preds_A, 'preds_B': preds_B,
        'proba_A': proba_A if has_proba else None,
        'proba_B': proba_B if has_proba else None,
    }

    # 3. Compute full metric set
    full_A = compute_full_metrics(y_all, preds_A, proba_A if has_proba else None)
    full_B = compute_full_metrics(y_all, preds_B, proba_B if has_proba else None)

    # Convenience aliases (keep existing variable names for bootstrap block below)
    acc_A   = full_A['accuracy'];   acc_B   = full_B['accuracy']
    f1_A    = full_A['f1_macro'];   f1_B    = full_B['f1_macro']
    kappa_A = full_A['cohen_kappa']; kappa_B = full_B['cohen_kappa']
    
    auroc_str_A = f"  AUROC={full_A['auroc']:.4f}" if 'auroc' in full_A else ""
    auroc_str_B = f"  AUROC={full_B['auroc']:.4f}" if 'auroc' in full_B else ""

    print(f"Aggregate Metrics (True Unseen):")
    print(f"  TRTR: Acc={acc_A:.4f}  F1={f1_A:.4f}  κ={kappa_A:.4f}  "
          f"BalAcc={full_A['balanced_accuracy']:.4f}{auroc_str_A}")
    print(f"  TSTR: Acc={acc_B:.4f}  F1={f1_B:.4f}  κ={kappa_B:.4f}  "
          f"BalAcc={full_B['balanced_accuracy']:.4f}{auroc_str_B}")
    print(f"  Δ: {acc_B - acc_A:+.4f}")

    # --- Salvage script (binary mapping) kept commented for reference ---
    # if np.max(preds_A) > 1: preds_A = np.where(preds_A < 3, 0, 1)
    # if np.max(preds_B) > 1: preds_B = np.where(preds_B < 3, 0, 1)
    
    # 4. Patient-level stratified bootstrap
    print(f"\nRunning patient-level bootstrap ({CFG['n_bootstrap']} iterations)...")
    boot_A, boot_B, boot_diffs = stratified_patient_bootstrap(
        preds_A, preds_B, y_all, patient_ids_all,
        n_boot=CFG['n_bootstrap'], seed=CFG['random_seed']
    )
    
    # 5. Bootstrap Statistics & Wilcoxon
    mean_diff   = np.mean(boot_diffs)
    median_diff = np.median(boot_diffs)
    
    ci_A_lo = np.quantile(boot_A, 0.025);  ci_A_hi = np.quantile(boot_A, 0.975)
    ci_B_lo = np.quantile(boot_B, 0.025);  ci_B_hi = np.quantile(boot_B, 0.975)
    ci_diff_lo = np.quantile(boot_diffs, 0.025)
    ci_diff_hi = np.quantile(boot_diffs, 0.975)

    n_pos      = np.sum(boot_diffs > 0)
    n_nonzero  = np.sum(boot_diffs != 0)
    
    if n_nonzero > 0:
        result     = binomtest(n_pos, n_nonzero, 0.5, alternative='greater')
        p_wilcoxon = result.pvalue
        z_score    = (n_pos - n_nonzero / 2) / np.sqrt(n_nonzero / 4)
        effect_r   = abs(z_score) / np.sqrt(len(y_all))
    else:
        p_wilcoxon = 1.0; effect_r = 0.0
        
    sig_corrected = p_wilcoxon < alpha_corrected
    
    # 6. McNemar Test
    correct_A = (preds_A == y_all)
    correct_B = (preds_B == y_all)
    n_11 = np.sum(correct_A & correct_B)
    n_10 = np.sum(correct_A & ~correct_B)
    n_01 = np.sum(~correct_A & correct_B)
    n_00 = np.sum(~correct_A & ~correct_B)
    
    if n_10 + n_01 > 0:
        p_mcnemar = mcnemar([[n_11, n_10], [n_01, n_00]], exact=False, correction=True).pvalue
    else:
        p_mcnemar = 1.0
        
    print(f"\nBootstrap Distribution (Δ = TSTR - TRTR):")
    print(f"  Mean:   {mean_diff:+.4f}  [95% CI: {ci_diff_lo:+.4f}, {ci_diff_hi:+.4f}]")
    print(f"Wilcoxon Test: p={p_wilcoxon:.4f} (Significant: {'✓' if sig_corrected else '✗'})")
    
    # Store results (existing keys preserved; new keys appended)
    results_all.append({
        "model": model_name, "acc_A": float(acc_A), "acc_B": float(acc_B),
        "acc_delta": float(acc_B - acc_A), "f1_A": float(f1_A), "f1_B": float(f1_B),
        "kappa_A": float(kappa_A), "kappa_B": float(kappa_B),
        "mean_diff": float(mean_diff), "median_diff": float(median_diff),
        "ci_A_lo": float(ci_A_lo), "ci_A_hi": float(ci_A_hi),
        "ci_B_lo": float(ci_B_lo), "ci_B_hi": float(ci_B_hi), 
        "ci_diff_lo": float(ci_diff_lo), "ci_diff_hi": float(ci_diff_hi),
        "p_wilcoxon": float(p_wilcoxon), "effect_r": float(effect_r),
        "sig_corrected": bool(sig_corrected), "p_mcnemar": float(p_mcnemar),
        "full_metrics_trtr": full_A,
        "full_metrics_tstr": full_B,
    })

## Section 6: Save & Summarize Results

In [ ]:
# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = Path(CFG['output_dir']) / f"statistical_evaluation_{timestamp}.json"

with open(json_path, 'w') as f:
    json.dump(results_all, f, indent=2)

print(f"✓ Results saved: {json_path}")

# Summary table
summary_df = pd.DataFrame([
    {
        "Model": r["model"],
        "Acc(TRTR)": f"{r['acc_A']:.4f}",
        "Acc(TSTR)": f"{r['acc_B']:.4f}",
        "Δ": f"{r['acc_delta']:+.4f}",
        "p-val": f"{r['p_wilcoxon']:.4f}",
        "Sig": "✓" if r['sig_corrected'] else "",
        "Effect": f"{r['effect_r']:.3f}",
    }
    for r in results_all
])

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(summary_df.to_string(index=False))

csv_path = Path(CFG['output_dir']) / f"statistical_summary_{timestamp}.csv"
summary_df.to_csv(csv_path, index=False)
print(f"\n✓ Summary saved: {csv_path}")

# ── Export full metrics as metrics_exp105.json / metrics_exp105.csv ─────────
metrics_exp105 = {}
csv_rows = []
for r in results_all:
    for regime, key in [("TRTR", "full_metrics_trtr"), ("TSTR", "full_metrics_tstr")]:
        fm = r[key]
        metrics_exp105.setdefault(r["model"], {})[regime] = fm
        flat = {"model": r["model"], "regime": regime}
        flat.update({k: v for k, v in fm.items() if k != "confusion_matrix"})
        flat["confusion_matrix"] = str(fm.get("confusion_matrix", ""))
        csv_rows.append(flat)

json_metrics_path = Path(CFG['output_dir']) / "metrics_exp105.json"
with open(json_metrics_path, 'w') as f:
    json.dump(metrics_exp105, f, indent=2)
print(f"✓ Full metrics saved: {json_metrics_path}")

csv_metrics_path = Path(CFG['output_dir']) / "metrics_exp105.csv"
pd.DataFrame(csv_rows).to_csv(csv_metrics_path, index=False)
print(f"✓ Full metrics CSV saved: {csv_metrics_path}")

In [ ]:
# ── Section 7: Patient / Slide-level aggregation ────────────────────────────
print(f"\n{'='*70}")
print("PATIENT-LEVEL AGGREGATION: Majority Vote vs Mean Probability")
print(f"{'='*70}")

slide_results_all = []

for model_name, stored in all_preds_store.items():
    preds_A = stored['preds_A']
    preds_B = stored['preds_B']
    proba_A = stored['proba_A']   # None if model had no proba support
    proba_B = stored['proba_B']

    slide_A = aggregate_by_patient(preds_A, y_all, patient_ids_all, proba_A)
    slide_B = aggregate_by_patient(preds_B, y_all, patient_ids_all, proba_B)
    sm_A    = compute_slide_metrics(slide_A)
    sm_B    = compute_slide_metrics(slide_B)

    def _fmt(sm, label):
        base = f"  {label} | Majority: Acc={sm['majority_accuracy']:.4f}  F1={sm['majority_f1_macro']:.4f}"
        if 'mean_prob_accuracy' in sm:
            base += f"  |  MeanProb: Acc={sm['mean_prob_accuracy']:.4f}  F1={sm['mean_prob_f1_macro']:.4f}"
        return base

    print(f"\nModel: {model_name}")
    print(_fmt(sm_A, "TRTR"))
    print(_fmt(sm_B, "TSTR"))

    slide_results_all.append({
        "model":              model_name,
        "trtr_slide_metrics": sm_A,
        "tstr_slide_metrics": sm_B,
        "trtr_slide_df":      slide_A.to_dict(orient='records'),
        "tstr_slide_df":      slide_B.to_dict(orient='records'),
    })

# ── Save slide-level metrics ─────────────────────────────────────────────────
slide_json_path = Path(CFG['output_dir']) / "slide_metrics_exp105.json"
slide_save = [
    {"model": r["model"], "trtr": r["trtr_slide_metrics"], "tstr": r["tstr_slide_metrics"]}
    for r in slide_results_all
]
with open(slide_json_path, 'w') as f:
    json.dump(slide_save, f, indent=2)
print(f"\n✓ Slide metrics saved: {slide_json_path}")

slide_csv_rows = []
for r in slide_results_all:
    for regime, key in [("TRTR", "trtr_slide_metrics"), ("TSTR", "tstr_slide_metrics")]:
        row = {"model": r["model"], "regime": regime}
        row.update(r[key])
        slide_csv_rows.append(row)

slide_csv_path = Path(CFG['output_dir']) / "slide_metrics_exp105.csv"
pd.DataFrame(slide_csv_rows).to_csv(slide_csv_path, index=False)
print(f"✓ Slide metrics CSV saved: {slide_csv_path}")

## Section 7: Interpretation

In [9]:
print(f"\n{'='*70}")
print("FINDINGS")
print(f"{'='*70}\n")

sig_models = [r for r in results_all if r['sig_corrected']]
improving = [r for r in results_all if r['acc_delta'] > 0]
degrading = [r for r in results_all if r['acc_delta'] < 0]

print(f"Statistically Significant Improvements (α = {alpha_corrected:.4f}):")
if sig_models:
    for r in sig_models:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")
else:
    print("  (None)")

print(f"\nImprovement Trend (not significant):")
for r in improving:
    if not r['sig_corrected']:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")

if degrading:
    print(f"\nDegradation:")
    for r in degrading:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")


FINDINGS

Statistically Significant Improvements (α = 0.0125):
  • Apollo      : Δ = +0.0077, p = 0.0000
  • RF_Standard : Δ = +0.0330, p = 0.0000
  • XGBoost     : Δ = +0.0182, p = 0.0000
  • 1D_CNN      : Δ = +0.0416, p = 0.0000

Improvement Trend (not significant):
